# Kapitola 1: Základní struktura promptu

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Experimentální playground](#example-playground)

## Nastavení

Spusťte následující nastavovací buňku, která načte API klíč a vytvoří pomocnou funkci `get_completion`.


In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lekce

Anthropic nabízí dvě API: starší [Text Completions API](https://docs.anthropic.com/claude/reference/complete_post) a aktuální [Messages API](https://docs.anthropic.com/claude/reference/messages_post). V tomto tutorialu budeme používat výhradně Messages API.

Minimální volání Claude přes Messages API vyžaduje tyto parametry:
- `model`: [API název modelu](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations), který chcete zavolat.

- `max_tokens`: maximální počet tokenů, které se mají vygenerovat před zastavením. Claude se může zastavit i před dosažením tohoto maxima. Tento parametr určuje pouze absolutní horní hranici počtu generovaných tokenů. Navíc jde o tvrdé zastavení, takže může způsobit, že Claude skončí uprostřed slova nebo věty.

- `messages`: pole vstupních zpráv. Naše modely jsou trénované pro práci se střídajícími se konverzačními tahy `user` a `assistant`. Při vytváření nové `Message` určíte předchozí konverzační tahy pomocí parametru `messages` a model potom vygeneruje další `Message` v konverzaci.
  - Každá vstupní zpráva musí být objekt s `role` a `content`. Můžete zadat jednu zprávu s rolí `user`, nebo více zpráv `user` a `assistant`, pokud se střídají. První zpráva musí vždy používat roli `user`.

Existují také volitelné parametry, například:
- `system`: system prompt, kterému se budeme věnovat níže.
  
- `temperature`: míra variability v odpovědi Claude. Pro tyto lekce a cvičení je `temperature` nastavená na 0.

Úplný seznam všech API parametrů najdete v [API dokumentaci](https://docs.anthropic.com/claude/reference/messages_post).


### Příklady

Podívejme se, jak Claude odpovídá na správně naformátované prompty. U každé z následujících buněk buňku spusťte (`Shift+Enter`) a odpověď Claude se zobrazí pod blokem.


In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Teď se podívejme na prompty, které neobsahují správné formátování Messages API. U těchto špatně naformátovaných promptů Messages API vrátí chybu.

Nejprve máme příklad volání Messages API, kterému v poli `messages` chybí pole `role` a `content`.


In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Zde je prompt, který nesprávně střídá role `user` a `assistant`.


In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Zprávy `user` a `assistant` se **MUSÍ střídat** a zprávy **MUSÍ začínat tahem `user`**. V promptu můžete mít více dvojic `user` a `assistant`, jako při simulaci vícekolové konverzace. Můžete také vložit slova do poslední zprávy `assistant`, aby Claude pokračoval od místa, kde jste skončili. K tomu se vrátíme v dalších kapitolách.

#### System prompty

Můžete také používat **system prompty**. System prompt je způsob, jak **poskytnout Claude kontext, instrukce a pravidla** ještě před tím, než mu v tahu `User` předložíte otázku nebo úkol.

Strukturálně system prompty existují odděleně od seznamu zpráv `user` a `assistant`, a proto patří do samostatného parametru `system`. Podívejte se na strukturu pomocné funkce `get_completion` v části [Nastavení](#setup) tohoto notebooku.

V tomto tutorialu všude, kde se může system prompt použít, najdete ve funkci completions pole `system`. Pokud system prompt použít nechcete, nastavte proměnnou `SYSTEM_PROMPT` na prázdný řetězec.


#### Příklad system promptu


In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Proč používat system prompt? **Dobře napsaný system prompt může zlepšit výkon Claude** různými způsoby, například zvýšit schopnost Claude dodržovat pravidla a instrukce. Více informací najdete v dokumentaci k tomu, [jak používat system prompty](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) s Claude.

Teď přejdeme ke cvičením. Pokud chcete experimentovat s prompty z lekce, aniž byste měnili obsah výše, sjeďte až na konec notebooku do části [**Example Playground**](#example-playground).


---

## Cvičení
- [Cvičení 1.1 - Počítání do tří](#exercise-11---counting-to-three)
- [Cvičení 1.2 - System Prompt](#exercise-12---system-prompt)


### Cvičení 1.1 - Počítání do tří
Pomocí správného formátování `user` / `assistant` upravte níže uvedený `PROMPT` tak, aby Claude **napočítal do tří**. Výstup také ukáže, zda je řešení správné.


In [ ]:
# Prompt - this is the only field you should change
PROMPT = "[Replace this text]"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### Cvičení 1.2 - System Prompt

Upravte `SYSTEM_PROMPT` tak, aby Claude odpovídal jako tříleté dítě.


In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### Gratulujeme!

Pokud jste vyřešili všechna cvičení až sem, jste připraveni přejít na další kapitolu. Hodně zdaru s promptováním.


---

## Example Playground

Toto je prostor, kde můžete volně experimentovat s příklady promptů z této lekce a upravovat je, abyste viděli, jak se tím mění odpovědi Claude.


In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))